In [8]:
# ============================================================
# 实验室作业：特征处理（第1部分）
# 数据集：Titanic (Kaggle)
# 任务：1. 缺失值处理  2. 分类特征编码  3. 数值特征归一化
# ============================================================

# 1. 导入所需库
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler

# 2. 加载数据集（公开数据，无需下载）
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Отображение основной информации о необработанных данных
print("="*50)
print("Информация об исходном наборе данных")
print("="*50)
print(f"Размер набора данных: {df.shape}")
print("\nПервые 5 строк данных:")
print(df.head())
print("\nСтатистика пропущенных значений:")
print(df.isnull().sum())


Информация об исходном наборе данных
Размер набора данных: (891, 12)

Первые 5 строк данных:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0       

In [9]:
# ============================================================
# Задание 1: Обработка пропущенных значений (исправленная версия, без предупреждений)
# ============================================================
print("\n" + "="*50)
print("Задание 1: Обработка пропущенных значений")
print("="*50)

# Создаём копию для обработки пропусков
df_missing = df.copy()

# 1.1 Пропуски в числовом признаке: Age -> заполняем медианой (исправлено)
age_median = df_missing['Age'].median()
df_missing['Age'] = df_missing['Age'].fillna(age_median)
print(f"Пропуски в Age заполнены медианой {age_median:.2f}")

# 1.2 Пропуски в категориальном признаке: Embarked -> заполняем модой (исправлено)
embarked_mode = df_missing['Embarked'].mode()[0]
df_missing['Embarked'] = df_missing['Embarked'].fillna(embarked_mode)
print(f"Пропуски в Embarked заполнены модой '{embarked_mode}'")

# 1.3 Необязательно: в Cabin слишком много пропусков, в этом эксперименте не обрабатываем
print(f"Доля пропусков в Cabin: {df_missing['Cabin'].isnull().mean()*100:.2f}% (в этом эксперименте не обрабатываем)")

# Проверяем, что пропуски обработаны
print("\nСтатистика пропусков после обработки:")
print(df_missing[['Age', 'Embarked']].isnull().sum())


Задание 1: Обработка пропущенных значений
Пропуски в Age заполнены медианой 28.00
Пропуски в Embarked заполнены модой 'S'
Доля пропусков в Cabin: 77.10% (в этом эксперименте не обрабатываем)

Статистика пропусков после обработки:
Age         0
Embarked    0
dtype: int64


In [10]:
# ============================================================
# Задание 2: Кодирование категориальных признаков
# ============================================================
print("\n" + "="*50)
print("Задание 2: Кодирование категориальных признаков")
print("="*50)

# Создаём копию для кодирования (на основе данных с обработанными пропусками)
df_encoded = df_missing.copy()

# 2.1 Label-кодирование (подходит для порядковых категорий, здесь показано для демонстрации)
le = LabelEncoder()
df_encoded['Sex_LabelEncoded'] = le.fit_transform(df_encoded['Sex'])
print(f"Сопоставление меток для Sex: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# 2.2 One-Hot кодирование (подходит для неупорядоченных категориальных признаков)
# Применяем One-Hot кодирование для Embarked
ohe = OneHotEncoder(sparse_output=False, drop='first')  # drop_first для избежания мультиколлинеарности
embarked_encoded = ohe.fit_transform(df_encoded[['Embarked']])
# Получаем названия закодированных столбцов
embarked_columns = [f"Embarked_{cat}" for cat in ohe.categories_[0][1:]]
embarked_df = pd.DataFrame(embarked_encoded, columns=embarked_columns, index=df_encoded.index)

# Добавляем столбцы с One-Hot кодированием
df_encoded = pd.concat([df_encoded, embarked_df], axis=1)

print(f"Новые столбцы после One-Hot кодирования Embarked: {embarked_columns}")
print("\nЧасть данных после кодирования (первые 5 строк, только столбцы, связанные с кодированием):")
print(df_encoded[['Sex', 'Sex_LabelEncoded', 'Embarked'] + embarked_columns].head())


Задание 2: Кодирование категориальных признаков
Сопоставление меток для Sex: {'female': np.int64(0), 'male': np.int64(1)}
Новые столбцы после One-Hot кодирования Embarked: ['Embarked_Q', 'Embarked_S']

Часть данных после кодирования (первые 5 строк, только столбцы, связанные с кодированием):
      Sex  Sex_LabelEncoded Embarked  Embarked_Q  Embarked_S
0    male                 1        S         0.0         1.0
1  female                 0        C         0.0         0.0
2  female                 0        S         0.0         1.0
3  female                 0        S         0.0         1.0
4    male                 1        S         0.0         1.0


In [11]:
# ============================================================
# Задание 3: Нормализация числовых признаков (исправленная версия)
# ============================================================
print("\n" + "="*50)
print("Задание 3: Нормализация числовых признаков")
print("="*50)

# Создаём копию для нормализации
df_normalized = df_encoded.copy()

# Выбираем числовые признаки для нормализации
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']

# 3.1 Стандартизация (Standardization) - среднее = 0, дисперсия = 1
scaler_std = StandardScaler()
std_values = scaler_std.fit_transform(df_normalized[numeric_features])
# Создаём новые столбцы для каждого стандартизированного признака
df_normalized['Age_std'] = std_values[:, 0]
df_normalized['Fare_std'] = std_values[:, 1]
df_normalized['SibSp_std'] = std_values[:, 2]
df_normalized['Parch_std'] = std_values[:, 3]

print("Стандартизация завершена, добавлены столбцы: Age_std, Fare_std, SibSp_std, Parch_std")

# 3.2 Min-Max нормализация - масштабирование в диапазон [0, 1]
scaler_minmax = MinMaxScaler()
minmax_values = scaler_minmax.fit_transform(df_normalized[numeric_features])
# Создаём новые столбцы для каждого нормализованного признака
df_normalized['Age_minmax'] = minmax_values[:, 0]
df_normalized['Fare_minmax'] = minmax_values[:, 1]
df_normalized['SibSp_minmax'] = minmax_values[:, 2]
df_normalized['Parch_minmax'] = minmax_values[:, 3]

print("Min-Max нормализация завершена, добавлены столбцы: Age_minmax, Fare_minmax, SibSp_minmax, Parch_minmax")

# Сравнение до и после нормализации
print("\nСтатистика исходных числовых признаков:")
print(df_normalized[numeric_features].describe().round(4))

print("\nСтатистика после стандартизации:")
std_cols = ['Age_std', 'Fare_std', 'SibSp_std', 'Parch_std']
print(df_normalized[std_cols].describe().round(4))

print("\nСтатистика после Min-Max нормализации:")
mm_cols = ['Age_minmax', 'Fare_minmax', 'SibSp_minmax', 'Parch_minmax']
print(df_normalized[mm_cols].describe().round(4))


Задание 3: Нормализация числовых признаков
Стандартизация завершена, добавлены столбцы: Age_std, Fare_std, SibSp_std, Parch_std
Min-Max нормализация завершена, добавлены столбцы: Age_minmax, Fare_minmax, SibSp_minmax, Parch_minmax

Статистика исходных числовых признаков:
            Age      Fare     SibSp     Parch
count  891.0000  891.0000  891.0000  891.0000
mean    29.3616   32.2042    0.5230    0.3816
std     13.0197   49.6934    1.1027    0.8061
min      0.4200    0.0000    0.0000    0.0000
25%     22.0000    7.9104    0.0000    0.0000
50%     28.0000   14.4542    0.0000    0.0000
75%     35.0000   31.0000    1.0000    0.0000
max     80.0000  512.3292    8.0000    6.0000

Статистика после стандартизации:
        Age_std  Fare_std  SibSp_std  Parch_std
count  891.0000  891.0000   891.0000   891.0000
mean     0.0000    0.0000     0.0000     0.0000
std      1.0006    1.0006     1.0006     1.0006
min     -2.2242   -0.6484    -0.4745    -0.4737
25%     -0.5657   -0.4891    -0.4745   

In [12]:
# ============================================================
# Итоговая проверка и вывод
# ============================================================
print("\n" + "="*50)
print("Эксперимент завершён — обзор итогового набора данных")
print("="*50)
print(f"Размер итогового набора данных: {df_normalized.shape}")
print("\nВсе названия столбцов:")
print(list(df_normalized.columns))

print("\nПервые 3 строки итоговых данных:")
print(df_normalized.head(3))


Эксперимент завершён — обзор итогового набора данных
Размер итогового набора данных: (891, 23)

Все названия столбцов:
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'Sex_LabelEncoded', 'Embarked_Q', 'Embarked_S', 'Age_std', 'Fare_std', 'SibSp_std', 'Parch_std', 'Age_minmax', 'Fare_minmax', 'SibSp_minmax', 'Parch_minmax']

Первые 3 строки итоговых данных:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   

   Parch            Ticket     Fare  ... Embarked_Q Embarked_S   Age_std  \
0      0         A/5 21171   7.2500  ... 